# Natural language -to-SQL with a Semantic Layer

![title](./images/semantic.jpg)

# NL-to-SQL with a Semantic Layer
## Decoupling Metadata from the LLM Prompt

In the previous notebook we embedded the full database schema directly in the system prompt. That works for a small schema — but it breaks down at scale:

| Problem | Schema-in-prompt | Semantic Layer |
|---------|-----------------|----------------|
| 100+ tables | Prompt too long, noisy | Only relevant tables retrieved |
| Schema change | Edit every prompt | Update one YAML file |
| Business users | Can't touch code | Edit YAML/metadata store |
| Prompt quality | Full schema = noise | Focused, precise context |

### How it works

```
User question
      │
      ▼
[ Embed question ]  ──────────────────────────────────────────┐
                                                               │
[ Semantic Layer ]  ← YAML metadata registry                  │
  (pre-embedded)    ← table descriptions + column docs        │
      │                                                        │
      ▼                                                        │
[ Retrieve top-N relevant tables ]  ◄── cosine similarity ◄───┘
      │
      ▼
[ Build focused system prompt ]  ← only matched tables
      │
      ▼
[ LLM → SQL ]
```

---

## Step 1: Setup

In [1]:
# !pip install openai numpy pyyaml

In [2]:
import getpass
import json
import math
import yaml
from openai import OpenAI

api_key = getpass.getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)
print("Client ready.")

Enter your OpenAI API key:  ········


Client ready.


## Step 2: Define the Semantic Layer

This is the central metadata registry — the single source of truth for your database schema. In production this would live in a YAML file or a dedicated tool (dbt, Cube.js, Atlan). Here we define it as a Python dict that mirrors what a YAML file would look like.

Key additions over a raw schema:
- **`description`** — business-friendly explanation used for semantic search
- **`tags`** — keywords that help match questions to tables
- **`column.description`** — explains what the field means in business terms
- **`column.values`** — documents valid codes/enums (critical for filters)

In [3]:
SEMANTIC_LAYER = {
    "database": "retail_db",
    "tables": {
        "ord_hdr": {
            "business_name": "Orders",
            "description": "One row per customer order. Use for order counts, order values, order status, revenue analysis, and sales trends.",
            "tags": ["orders", "sales", "revenue", "transactions", "status", "pending", "completed", "cancelled"],
            "columns": {
                "ord_id":   {"type": "INTEGER", "primary_key": True,  "description": "Unique order identifier"},
                "cust_ref": {"type": "INTEGER", "foreign_key": "cust_mst.cust_ref", "description": "Customer reference, links to the customer master table"},
                "ord_dt":   {"type": "DATE",    "description": "Date the order was placed"},
                "tot_amt":  {"type": "DECIMAL", "description": "Total order amount in USD"},
                "sts_cd":   {"type": "VARCHAR", "description": "Order status code",
                             "values": {"COMP": "completed", "PEND": "pending", "CNCL": "cancelled"}}
            }
        },
        "cust_mst": {
            "business_name": "Customers",
            "description": "One row per customer. Use for customer profiles, regions, acquisition sources, and customer-level aggregations.",
            "tags": ["customers", "users", "region", "acquisition", "segments", "geography", "europe", "north america", "apac", "referral", "organic", "paid"],
            "columns": {
                "cust_ref": {"type": "INTEGER", "primary_key": True,  "description": "Unique customer reference number"},
                "cust_nm":  {"type": "VARCHAR", "description": "Customer full name"},
                "rgn_cd":   {"type": "VARCHAR", "description": "Region code",
                             "values": {"NA": "North America", "EU": "Europe", "APAC": "Asia Pacific"}},
                "acq_src":  {"type": "VARCHAR", "description": "Channel through which the customer was acquired",
                             "values": {"ORGANIC": "organic search", "PAID": "paid advertising", "REFERRAL": "referral"}},
                "acq_dt":   {"type": "DATE",    "description": "Date the customer was first acquired"}
            }
        },
        "prd_cat": {
            "business_name": "Products",
            "description": "One row per product. Use for product listings, prices, categories, and stock/inventory levels.",
            "tags": ["products", "inventory", "stock", "price", "catalog", "categories", "items"],
            "columns": {
                "prd_id":   {"type": "INTEGER", "primary_key": True,  "description": "Unique product identifier"},
                "prd_nm":   {"type": "VARCHAR", "description": "Product name"},
                "cat_cd":   {"type": "VARCHAR", "description": "Product category code"},
                "unit_prc": {"type": "DECIMAL", "description": "Unit price per item in USD"},
                "stk_qty":  {"type": "INTEGER", "description": "Current stock quantity available"}
            }
        },
        "ord_dtl": {
            "business_name": "Order Line Items",
            "description": "One row per product within an order. Use for product-level order analysis, quantities ordered, and line-level revenue.",
            "tags": ["line items", "order details", "quantity", "products ordered", "basket"],
            "columns": {
                "ord_id":   {"type": "INTEGER", "foreign_key": "ord_hdr.ord_id",  "description": "Order reference, links to order header"},
                "prd_id":   {"type": "INTEGER", "foreign_key": "prd_cat.prd_id",  "description": "Product reference, links to product catalog"},
                "qty_ord":  {"type": "INTEGER", "description": "Quantity of this product ordered"},
                "ln_amt":   {"type": "DECIMAL", "description": "Total line amount (qty x unit price) in USD"}
            }
        }
    }
}

print(f"Semantic layer loaded: {len(SEMANTIC_LAYER['tables'])} tables")
for name, table in SEMANTIC_LAYER["tables"].items():
    print(f"  {name:12s} ({table['business_name']}) — {len(table['columns'])} columns")

Semantic layer loaded: 4 tables
  ord_hdr      (Orders) — 5 columns
  cust_mst     (Customers) — 5 columns
  prd_cat      (Products) — 5 columns
  ord_dtl      (Order Line Items) — 4 columns


## Step 3: Pre-embed the Semantic Layer

We embed each table's description + tags + column descriptions once and store them. At query time we only need to embed the user question, then compare against these stored vectors.

In [4]:
EMBEDDING_MODEL = "text-embedding-3-small"

def embed(text: str) -> list[float]:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding

def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x ** 2 for x in a))
    mag_b = math.sqrt(sum(x ** 2 for x in b))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0


def build_table_embedding_text(table_name: str, table_def: dict) -> str:
    """Flatten a table definition into a single string for embedding."""
    parts = [
        f"Table: {table_name} ({table_def['business_name']})",
        f"Description: {table_def['description']}",
        f"Tags: {', '.join(table_def['tags'])}",
        "Columns:"
    ]
    for col, meta in table_def["columns"].items():
        col_text = f"  {col}: {meta['description']}"
        if "values" in meta:
            col_text += " (" + ", ".join(f"{k}={v}" for k, v in meta["values"].items()) + ")"
        parts.append(col_text)
    return "\n".join(parts)


# Pre-embed all tables
print("Pre-embedding semantic layer...")
TABLE_EMBEDDINGS = {}
for table_name, table_def in SEMANTIC_LAYER["tables"].items():
    text = build_table_embedding_text(table_name, table_def)
    TABLE_EMBEDDINGS[table_name] = {"text": text, "embedding": embed(text)}
    print(f"  Embedded: {table_name}")

print("\nDone. Semantic layer is ready for queries.")

Pre-embedding semantic layer...
  Embedded: ord_hdr
  Embedded: cust_mst
  Embedded: prd_cat
  Embedded: ord_dtl

Done. Semantic layer is ready for queries.


## Step 4: The Semantic Retrieval Engine

Given a user question, this finds the most relevant tables by cosine similarity and builds a focused prompt — only the tables the LLM actually needs.

In [5]:
def retrieve_relevant_tables(question: str, top_n: int = 2, threshold: float = 0.35) -> list[str]:
    """Return table names most relevant to the question."""
    question_embedding = embed(question)
    scores = [
        (name, cosine_similarity(question_embedding, data["embedding"]))
        for name, data in TABLE_EMBEDDINGS.items()
    ]
    scores.sort(key=lambda x: x[1], reverse=True)
    # Always include at least top_n, and any others above threshold
    selected = [name for name, score in scores[:top_n]]
    for name, score in scores[top_n:]:
        if score >= threshold:
            selected.append(name)
    return selected, scores


def build_prompt_from_semantic_layer(table_names: list[str]) -> str:
    """Construct a focused system prompt from only the retrieved tables."""
    lines = [
        "You are a SQL expert. Convert the user's natural language question into a SQL query.",
        "Return ONLY the SQL query, no explanation.",
        "",
        "Relevant tables for this query:",
    ]
    for table_name in table_names:
        table_def = SEMANTIC_LAYER["tables"][table_name]
        lines.append(f"\nTable: {table_name} ({table_def['business_name']})")
        for col, meta in table_def["columns"].items():
            col_line = f"  - {col} ({meta['type']}): {meta['description']}"
            if "foreign_key" in meta:
                col_line += f" → FK: {meta['foreign_key']}"
            if "values" in meta:
                col_line += " | values: " + ", ".join(f"{k}={v}" for k, v in meta["values"].items())
            lines.append(col_line)
    return "\n".join(lines)


# Demo the retrieval
sample_question = "Show me the top 5 customers by total order amount"
tables, scores = retrieve_relevant_tables(sample_question)

print(f"Question: \"{sample_question}\"")
print("\nSimilarity scores:")
for name, score in scores:
    marker = "  ◄ selected" if name in tables else ""
    print(f"  {name:12s}  {score:.3f}{marker}")
print(f"\nTables selected for prompt: {tables}")
print("\n" + "=" * 60)
print("Generated system prompt:")
print("=" * 60)
print(build_prompt_from_semantic_layer(tables))

Question: "Show me the top 5 customers by total order amount"

Similarity scores:
  ord_hdr       0.413  ◄ selected
  cust_mst      0.354  ◄ selected
  ord_dtl       0.342
  prd_cat       0.254

Tables selected for prompt: ['ord_hdr', 'cust_mst']

Generated system prompt:
You are a SQL expert. Convert the user's natural language question into a SQL query.
Return ONLY the SQL query, no explanation.

Relevant tables for this query:

Table: ord_hdr (Orders)
  - ord_id (INTEGER): Unique order identifier
  - cust_ref (INTEGER): Customer reference, links to the customer master table → FK: cust_mst.cust_ref
  - ord_dt (DATE): Date the order was placed
  - tot_amt (DECIMAL): Total order amount in USD
  - sts_cd (VARCHAR): Order status code | values: COMP=completed, PEND=pending, CNCL=cancelled

Table: cust_mst (Customers)
  - cust_ref (INTEGER): Unique customer reference number
  - cust_nm (VARCHAR): Customer full name
  - rgn_cd (VARCHAR): Region code | values: NA=North America, EU=Europe, AP

## Step 5: The Two Agents — Hardcoded vs. Semantic Layer

We now define both approaches so we can compare them directly.

In [6]:
LLM_MODEL = "gpt-4o-mini"

# --- Approach A: hardcoded schema in the prompt (old way) ---
HARDCODED_SCHEMA_PROMPT = """
You are a SQL expert for a retail company. Convert the user's natural language question into a SQL query.
Return ONLY the SQL query, no explanation.

Database schema:

Table: ord_hdr (order headers)
  - ord_id      : INTEGER PRIMARY KEY
  - cust_ref    : INTEGER (foreign key to cust_mst.cust_ref)
  - ord_dt      : DATE (order date)
  - tot_amt     : DECIMAL (total order amount)
  - sts_cd      : VARCHAR (status code: 'COMP'=completed, 'PEND'=pending, 'CNCL'=cancelled)

Table: cust_mst (customer master)
  - cust_ref    : INTEGER PRIMARY KEY
  - cust_nm     : VARCHAR (customer full name)
  - rgn_cd      : VARCHAR (region code: 'NA'=North America, 'EU'=Europe, 'APAC'=Asia Pacific)
  - acq_src     : VARCHAR (acquisition source: 'ORGANIC', 'PAID', 'REFERRAL')
  - acq_dt      : DATE (date customer was acquired)

Table: prd_cat (product catalog)
  - prd_id      : INTEGER PRIMARY KEY
  - prd_nm      : VARCHAR (product name)
  - cat_cd      : VARCHAR (category code)
  - unit_prc    : DECIMAL (unit price)
  - stk_qty     : INTEGER (stock quantity)

Table: ord_dtl (order line items)
  - ord_id      : INTEGER (foreign key to ord_hdr.ord_id)
  - prd_id      : INTEGER (foreign key to prd_cat.prd_id)
  - qty_ord     : INTEGER (quantity ordered)
  - ln_amt      : DECIMAL (line amount)
"""

def agent_hardcoded(question: str) -> str:
    """NL-to-SQL using a fixed, hardcoded schema in the prompt."""
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": HARDCODED_SCHEMA_PROMPT},
            {"role": "user",   "content": question}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()


# --- Approach B: semantic layer retrieval (new way) ---
def agent_semantic_layer(question: str, verbose: bool = False) -> str:
    """NL-to-SQL using dynamic prompt built from the semantic layer."""
    tables, scores = retrieve_relevant_tables(question)
    system_prompt = build_prompt_from_semantic_layer(tables)

    if verbose:
        print(f"  Tables retrieved: {tables}")
        print(f"  Prompt length: {len(system_prompt)} chars  (vs {len(HARDCODED_SCHEMA_PROMPT)} chars hardcoded)")

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()


print("Both agents ready.")
print(f"Hardcoded prompt size : {len(HARDCODED_SCHEMA_PROMPT):,} characters")

Both agents ready.
Hardcoded prompt size : 1,291 characters


## Step 6: Side-by-Side Comparison

In [7]:
test_questions = [
    "Show me the top 5 customers by total order amount",
    "Which products have less than 10 items in stock?",
    "How many orders were completed in each region?",
    "What is the average order value for customers acquired through paid channels?",
    "Show products that have never been ordered",
]

print("SIDE-BY-SIDE COMPARISON")
print("=" * 80)

for q in test_questions:
    sql_hardcoded = agent_hardcoded(q)
    sql_semantic  = agent_semantic_layer(q, verbose=True)

    print(f"\nQUESTION: {q}")
    print()
    print("  [HARDCODED PROMPT — full schema every time]")
    for line in sql_hardcoded.splitlines():
        print(f"    {line}")
    print()
    print("  [SEMANTIC LAYER — focused prompt, only relevant tables]")
    for line in sql_semantic.splitlines():
        print(f"    {line}")
    print("\n" + "-" * 80)

SIDE-BY-SIDE COMPARISON
  Tables retrieved: ['ord_hdr', 'cust_mst']
  Prompt length: 944 chars  (vs 1291 chars hardcoded)

QUESTION: Show me the top 5 customers by total order amount

  [HARDCODED PROMPT — full schema every time]
    ```sql
    SELECT cust_ref, SUM(tot_amt) AS total_order_amount
    FROM ord_hdr
    GROUP BY cust_ref
    ORDER BY total_order_amount DESC
    LIMIT 5;
    ```

  [SEMANTIC LAYER — focused prompt, only relevant tables]
    ```sql
    SELECT c.cust_nm, SUM(o.tot_amt) AS total_order_amount
    FROM cust_mst c
    JOIN ord_hdr o ON c.cust_ref = o.cust_ref
    GROUP BY c.cust_nm
    ORDER BY total_order_amount DESC
    LIMIT 5;
    ```

--------------------------------------------------------------------------------
  Tables retrieved: ['prd_cat', 'ord_dtl']
  Prompt length: 747 chars  (vs 1291 chars hardcoded)

QUESTION: Which products have less than 10 items in stock?

  [HARDCODED PROMPT — full schema every time]
    ```sql
    SELECT prd_id, prd_nm, stk_qt

## Step 7: The Maintainability Win — Add a New Table

Imagine your company adds a **returns** table. With the hardcoded approach you'd hunt down every prompt file and edit it. With the semantic layer you update **one place** — the registry — and every agent immediately knows about it.

Let's add a `rtn_hdr` (returns) table and watch the agent use it instantly.

In [8]:
# Add a new table to the semantic layer
SEMANTIC_LAYER["tables"]["rtn_hdr"] = {
    "business_name": "Returns",
    "description": "One row per product return request. Use for return rates, refund amounts, return reasons, and return trends.",
    "tags": ["returns", "refunds", "returned", "complaints", "defective", "exchange"],
    "columns": {
        "rtn_id":    {"type": "INTEGER", "primary_key": True,            "description": "Unique return identifier"},
        "ord_id":    {"type": "INTEGER", "foreign_key": "ord_hdr.ord_id", "description": "Original order reference"},
        "prd_id":    {"type": "INTEGER", "foreign_key": "prd_cat.prd_id", "description": "Product that was returned"},
        "rtn_dt":    {"type": "DATE",    "description": "Date the return was logged"},
        "rtn_rsn":   {"type": "VARCHAR", "description": "Return reason code",
                     "values": {"DEFECT": "defective item", "WRONG": "wrong item sent", "NOMATCH": "did not match description", "CHANGE": "customer changed mind"}},
        "rfnd_amt":  {"type": "DECIMAL", "description": "Refund amount issued in USD"}
    }
}

# Re-embed the new table (only this one — existing embeddings are unchanged)
new_table = "rtn_hdr"
text = build_table_embedding_text(new_table, SEMANTIC_LAYER["tables"][new_table])
TABLE_EMBEDDINGS[new_table] = {"text": text, "embedding": embed(text)}
print(f"Added '{new_table}' to semantic layer and embedding index.")
print(f"Total tables now: {len(SEMANTIC_LAYER['tables'])}")

Added 'rtn_hdr' to semantic layer and embedding index.
Total tables now: 5


In [9]:
# The agent now knows about returns — no prompt editing needed
return_questions = [
    "What is the total refund amount by return reason?",
    "Which products are returned most often?",
    "Show me the return rate for defective items this year",
]

print("TESTING WITH NEW 'rtn_hdr' TABLE")
print("(No changes to agent code — only the semantic layer was updated)")
print("=" * 70)

for q in return_questions:
    tables, scores = retrieve_relevant_tables(q)
    sql = agent_semantic_layer(q, verbose=True)
    print(f"\nQ: {q}")
    print(f"SQL:\n{sql}")
    print("-" * 40)

TESTING WITH NEW 'rtn_hdr' TABLE
(No changes to agent code — only the semantic layer was updated)
  Tables retrieved: ['rtn_hdr', 'ord_hdr']
  Prompt length: 993 chars  (vs 1291 chars hardcoded)

Q: What is the total refund amount by return reason?
SQL:
```sql
SELECT rtn_rsn, SUM(rfnd_amt) AS total_refund
FROM rtn_hdr
GROUP BY rtn_rsn;
```
----------------------------------------
  Tables retrieved: ['rtn_hdr', 'prd_cat']
  Prompt length: 891 chars  (vs 1291 chars hardcoded)

Q: Which products are returned most often?
SQL:
```sql
SELECT p.prd_nm, COUNT(r.rtn_id) AS return_count
FROM rtn_hdr r
JOIN prd_cat p ON r.prd_id = p.prd_id
GROUP BY p.prd_nm
ORDER BY return_count DESC;
```
----------------------------------------
  Tables retrieved: ['rtn_hdr', 'ord_dtl']
  Prompt length: 956 chars  (vs 1291 chars hardcoded)

Q: Show me the return rate for defective items this year
SQL:
```sql
SELECT 
    COUNT(rtn_id) * 100.0 / NULLIF(SUM(qty_ord), 0) AS return_rate
FROM 
    rtn_hdr 
JOIN 
    

## Step 8: Visualise What the Retrieval "Sees"

This cell shows exactly what context the LLM receives for any question — useful for debugging why the agent generated a particular query.

In [10]:
def explain_retrieval(question: str):
    tables, scores = retrieve_relevant_tables(question)
    prompt = build_prompt_from_semantic_layer(tables)

    print(f"Question : {question}")
    print(f"\nSimilarity scores:")
    for name, score in scores:
        bar = "█" * int(score * 40)
        marker = " ◄" if name in tables else ""
        print(f"  {name:12s}  {score:.3f}  {bar}{marker}")
    print(f"\nTables in prompt : {tables}")
    print(f"Prompt size      : {len(prompt):,} chars  (full schema = {len(HARDCODED_SCHEMA_PROMPT):,} chars)")
    print(f"\nSystem prompt sent to LLM:")
    print("-" * 60)
    print(prompt)
    print("-" * 60)


explain_retrieval("Which products are returned most often?")

Question : Which products are returned most often?

Similarity scores:
  rtn_hdr       0.387  ███████████████ ◄
  prd_cat       0.326  █████████████ ◄
  ord_dtl       0.280  ███████████
  ord_hdr       0.242  █████████
  cust_mst      0.240  █████████

Tables in prompt : ['rtn_hdr', 'prd_cat']
Prompt size      : 891 chars  (full schema = 1,291 chars)

System prompt sent to LLM:
------------------------------------------------------------
You are a SQL expert. Convert the user's natural language question into a SQL query.
Return ONLY the SQL query, no explanation.

Relevant tables for this query:

Table: rtn_hdr (Returns)
  - rtn_id (INTEGER): Unique return identifier
  - ord_id (INTEGER): Original order reference → FK: ord_hdr.ord_id
  - prd_id (INTEGER): Product that was returned → FK: prd_cat.prd_id
  - rtn_dt (DATE): Date the return was logged
  - rtn_rsn (VARCHAR): Return reason code | values: DEFECT=defective item, WRONG=wrong item sent, NOMATCH=did not match description, CHANGE=c

## Step 9: Interactive Agent

In [11]:
print("Semantic Layer NL-to-SQL Agent")
print(f"Tables available: {', '.join(SEMANTIC_LAYER['tables'].keys())}")
print("Type 'explain <question>' to see which tables were retrieved.")
print("Type 'quit' to exit.")
print("-" * 60)

while True:
    user_input = input("\nYour question: ").strip()
    if not user_input or user_input.lower() == "quit":
        print("Goodbye!")
        break

    if user_input.lower().startswith("explain "):
        explain_retrieval(user_input[8:].strip())
    else:
        sql = agent_semantic_layer(user_input, verbose=True)
        print(f"\nGenerated SQL:\n{sql}")

Semantic Layer NL-to-SQL Agent
Tables available: ord_hdr, cust_mst, prd_cat, ord_dtl, rtn_hdr
Type 'explain <question>' to see which tables were retrieved.
Type 'quit' to exit.
------------------------------------------------------------



Your question:  give me top 5 customers by revenu


  Tables retrieved: ['cust_mst', 'ord_hdr']
  Prompt length: 944 chars  (vs 1291 chars hardcoded)

Generated SQL:
```sql
SELECT cust_mst.cust_ref, cust_mst.cust_nm, SUM(ord_hdr.tot_amt) AS total_revenue
FROM cust_mst
JOIN ord_hdr ON cust_mst.cust_ref = ord_hdr.cust_ref
WHERE ord_hdr.sts_cd = 'COMP'
GROUP BY cust_mst.cust_ref, cust_mst.cust_nm
ORDER BY total_revenue DESC
LIMIT 5;
```



Your question:  quit


Goodbye!


---
## Summary

| | Hardcoded Schema in Prompt | Semantic Layer |
|--|--|--|
| **Schema storage** | Scattered across prompt files | Centralised YAML / registry |
| **Prompt size** | Full schema every call | Only relevant tables |
| **Add a new table** | Edit every prompt file | Add one entry to the registry |
| **Scales to 100+ tables** | No — prompt overflows | Yes — retrieval selects what's needed |
| **Business users can maintain** | No — requires code access | Yes — edit YAML |
| **Debuggability** | Opaque | `explain` shows exactly what LLM saw |

### What comes next?
In a production system the semantic layer would live in a dedicated tool:
- **[dbt](https://docs.getdbt.com/docs/build/semantic-layer)** — define metrics and dimensions alongside your SQL models
- **[Cube.js](https://cube.dev)** — headless BI semantic layer with API
- **[Atlan](https://atlan.com)** / **[Alation](https://alation.com)** — enterprise data catalog with business glossary
- **[OpenMetadata](https://open-metadata.org)** — open-source metadata platform

The retrieval pattern shown here (embed → cosine search → dynamic prompt) is the same regardless of where the metadata lives.